# Preprocessing: Align Assets & Compute Log Returns
1. Load raw CSVs (BTC, S&P 500, Gold)
2. Resample BTC to business days (last close of each business day)
3. Inner-join all three on the S&P 500 trading calendar
4. Compute log returns and save to `data/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load raw CSVs

In [2]:
DATA_DIR = Path("../data")

def load_raw(filename: str) -> pd.Series:
    """Load a yfinance-style CSV, return a Series of Close prices indexed by Date."""
    df = pd.read_csv(
        DATA_DIR / filename,
        header=[0, 1],   # two-row header: Price row + Ticker row
        index_col=0,
        parse_dates=True,
        skiprows=[2],    # skip the empty 'Date' label row
    )
    # Keep only the Close column and drop the multi-level header
    close = df["Close"].squeeze()
    close.index.name = "Date"
    close = pd.to_numeric(close, errors="coerce")
    return close

btc   = load_raw("raw_btc.csv").rename("BTC")
sp500 = load_raw("raw_sp500.csv").rename("SP500")
gold  = load_raw("raw_gold.csv").rename("Gold")

print("BTC  :", btc.index.min().date(), "→", btc.index.max().date(), f"({len(btc)} rows)")
print("SP500:", sp500.index.min().date(), "→", sp500.index.max().date(), f"({len(sp500)} rows)")
print("Gold :", gold.index.min().date(), "→", gold.index.max().date(), f"({len(gold)} rows)")

BTC  : 2017-01-01 → 2026-02-28 (3346 rows)
SP500: 2017-01-03 → 2026-02-27 (2301 rows)
Gold : 2017-01-03 → 2026-02-27 (2302 rows)


## 2. Resample BTC to business days (last close of each business day)

In [3]:
# BTC trades 24/7 — take the last available close within each business day
btc_bday = btc.resample("B").last().dropna()

print(f"BTC after resample: {len(btc_bday)} business days")
btc_bday.head()

BTC after resample: 2391 business days


Date
2016-12-30     998.325012
2017-01-02    1021.750000
2017-01-03    1043.839966
2017-01-04    1154.729980
2017-01-05    1013.380005
Freq: B, Name: BTC, dtype: float64

## 3. Inner-join on the S&P 500 trading calendar

In [4]:
# S&P 500 already follows the NYSE calendar (excludes weekends & US holidays).
# Inner-join forces all series to share exactly those dates.
prices = pd.concat([sp500, btc_bday, gold], axis=1, join="inner")
prices.index.freq = None  # remove any inferred freq to avoid downstream warnings

print(f"Aligned dataset: {len(prices)} trading days  |  {prices.shape[1]} assets")
print(f"Date range: {prices.index.min().date()} → {prices.index.max().date()}")
prices.head()

Aligned dataset: 2300 trading days  |  3 assets
Date range: 2017-01-03 → 2026-02-27


,SP500,BTC,Gold
Date,,,
2017-01-03,2257.830078,1043.839966,1160.400024
2017-01-04,2270.750000,1154.729980,1163.800049
2017-01-05,2269.000000,1013.380005,1179.699951
2017-01-06,2276.979980,911.198975,1171.900024
2017-01-09,2268.899902,902.828003,1183.500000


In [5]:
# Sanity check: no NaNs after the inner join
assert prices.isna().sum().sum() == 0, "Unexpected NaNs in aligned prices!"
prices.describe()

,SP500,BTC,Gold
count,2300.000000,2300.000000,2300.000000
mean,4007.694299,34371.689563,1946.741041
std,1276.040524,32489.939208,770.058816
min,2237.399902,777.757019,1160.400024
25%,2852.114990,8145.441650,1333.300018
50%,3929.470093,22966.891602,1799.049988
75%,4689.185181,54828.918945,2011.050018
max,6978.600098,124752.531250,5318.399902


## 4. Compute log returns

In [6]:
log_returns = np.log(prices / prices.shift(1)).dropna()

print(f"Log-return matrix: {log_returns.shape}")
log_returns.describe()

Log-return matrix: (2299, 3)


,SP500,BTC,Gold
count,2299.000000,2299.000000,2299.000000
mean,0.000485,0.001810,0.000655
std,0.011703,0.043256,0.010198
min,-0.127652,-0.464730,-0.120657
25%,-0.003765,-0.017296,-0.003981
50%,0.000819,0.001084,0.000813
75%,0.005881,0.022112,0.005924
max,0.090895,0.242686,0.059054


## 5. Save outputs

In [7]:
prices.to_csv(DATA_DIR / "aligned_prices.csv")
log_returns.to_csv(DATA_DIR / "log_returns.csv")

print("Saved:")
print(f"  {DATA_DIR / 'aligned_prices.csv'}  ({len(prices)} rows)")
print(f"  {DATA_DIR / 'log_returns.csv'}  ({len(log_returns)} rows)")

Saved:
  ../data/aligned_prices.csv  (2300 rows)
  ../data/log_returns.csv  (2299 rows)
